# Research Project Context
* **Author:** Shalev Yaacov
* **Date:** 2026-06-28
* **Research Project Context and Objective:** Generate a UMAP visualization of the trained Three-Tower fusion model's learned 64-dim embedding space. Training genes (434) are colored by module label; top novel candidates (high-reliability, full PPI coverage) are overlaid as star markers. The plot reveals how the model organizes known and predicted IRD genes.

Instructions for each subsequent editor of the script:
Code Style: Use clean and efficient Python code, meaningful variable names, and inline comments in English only.
Logging: Include print statements to track the progress of long-running steps.
Ensure the script can be run "Top to Bottom" without errors.

In [ ]:
# Configuration
script_name = "umap_visualization"

# Setup standard libraries, paths, and logging
import os
import sys
import logging
from datetime import datetime
import pandas as pd
import numpy as np

# Configuration
INPUT_DIR = "../input"
OUTPUT_DIR = "../output/figures"
CHECKPOINT_DIR = "../checkpoints"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Configure timestamp and logging
timestamp = datetime.now().strftime("%y%m%d_%H%M")
log_path = os.path.join(OUTPUT_DIR, f'run_{timestamp}.log')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s', handlers=[logging.FileHandler(log_path), logging.StreamHandler()])
logging.info("Environment initialized. Ready for processing.")

In [ ]:
# Install umap-learn and adjustText if not already available
import importlib

for pkg_import, pkg_pip in [("umap", "umap-learn"), ("adjustText", "adjustText")]:
    try:
        importlib.import_module(pkg_import)
        logging.info(f"{pkg_pip} is already installed.")
    except ImportError:
        logging.info(f"Installing {pkg_pip}...")
        !pip install {pkg_pip}
        logging.info(f"{pkg_pip} installed successfully.")


## Data Loading
* **What:** Load the training set, unlabeled set, and concordant candidates CSV.
* **Why:** To obtain the feature arrays (npp, esm2, ppi), gene names, labels, and the novel candidate list.
* **Variables:** Input (NPZ files, CSV) -> Output (train_data, unlabeled_data, candidates_df).
* **Logging:** Accurate printing - which file is loaded + exact path.

In [ ]:
# Load training set (434 genes)
train_path = os.path.join(INPUT_DIR, "training_set_npp_esm2_ppi.npz")
train_data = np.load(train_path, allow_pickle=True)
logging.info(f"Loaded training set from {train_path}")
logging.info(f"  gene_names: {train_data['gene_names'].shape}, labels: {train_data['labels'].shape}")
logging.info(f"  npp: {train_data['npp'].shape}, esm2: {train_data['esm2'].shape}, ppi: {train_data['ppi'].shape}")

# Load unlabeled set (19,573 genes)
unlabeled_path = os.path.join(INPUT_DIR, "unlabeled_set_npp_esm2_ppi.npz")
unlabeled_data = np.load(unlabeled_path, allow_pickle=True)
logging.info(f"Loaded unlabeled set from {unlabeled_path}")
logging.info(f"  gene_names: {unlabeled_data['gene_names'].shape}")
logging.info(f"  npp: {unlabeled_data['npp'].shape}, esm2: {unlabeled_data['esm2'].shape}, ppi: {unlabeled_data['ppi'].shape}")

# Load concordant candidates
candidates_path = os.path.join("..", "output", "inference", "concordant_candidates.csv")
candidates_df = pd.read_csv(candidates_path)
logging.info(f"Loaded concordant candidates from {candidates_path} — {len(candidates_df)} rows")

## Filter Novel Candidates
* **Objective (What):** Select the top 5 novel candidates per predicted module from the concordant candidates.
* **Rationale (Why):** We focus on high-reliability genes with full PPI coverage to overlay only the most confident predictions on the UMAP.
* **Logical Steps:**
    1. Filter to reliability_tier == 'high' AND ppi_coverage == 1.
    2. Sort by mean_confidence descending within each predicted_module.
    3. Take top 5 genes per module.
* **I/O Variables:** Input: candidates_df -> Output: novel_df (filtered dataframe)

In [ ]:
# Filter to high-reliability candidates with full PPI coverage
novel_df = candidates_df[
    (candidates_df["reliability_tier"] == "high") &
    (candidates_df["ppi_coverage"] == 1)
].copy()
logging.info(f"After filtering (high reliability, ppi_coverage=1): {len(novel_df)} candidates")

# Take top 5 per predicted_module by mean_confidence descending
novel_df = (
    novel_df
    .sort_values("mean_confidence", ascending=False)
    .groupby("predicted_module")
    .head(5)
    .reset_index(drop=True)
)
logging.info(f"Selected top-5 per module: {len(novel_df)} novel candidates total")
logging.info(f"Modules represented: {sorted(novel_df['predicted_module'].unique())}")
logging.info(f"Novel candidate genes: {novel_df['gene_name'].tolist()}")

## Extract Novel Candidate Features from the Unlabeled Set
* **Objective (What):** Pull npp, esm2, and ppi feature rows for the selected novel candidates.
* **Rationale (Why):** The model needs these features as input to compute embeddings for the novel candidates.
* **Logical Steps:**
    1. Build a gene_name-to-index map for the unlabeled set.
    2. Look up indices for each novel candidate gene.
    3. Slice the feature arrays.
* **I/O Variables:** Input: unlabeled_data, novel_df -> Output: novel_npp, novel_esm2, novel_ppi (numpy arrays)

In [ ]:
# Build gene_name -> index mapping for the unlabeled set
unlabeled_gene_names = unlabeled_data["gene_names"]
gene_to_idx = {name: i for i, name in enumerate(unlabeled_gene_names)}

# Find indices for novel candidates
novel_genes = novel_df["gene_name"].tolist()
novel_indices = [gene_to_idx[g] for g in novel_genes]
logging.info(f"Matched {len(novel_indices)} / {len(novel_genes)} novel candidates in unlabeled set")

# Extract feature arrays for novel candidates
novel_npp = unlabeled_data["npp"][novel_indices]
novel_esm2 = unlabeled_data["esm2"][novel_indices]
novel_ppi = unlabeled_data["ppi"][novel_indices]
logging.info(f"Novel candidate features — npp: {novel_npp.shape}, esm2: {novel_esm2.shape}, ppi: {novel_ppi.shape}")

## Compute Fold-Averaged Embeddings
* **Objective (What):** Extract 64-dim embeddings from each of the 5 cross-validation folds and average them.
* **Rationale (Why):** Averaging over folds produces a more stable and representative embedding that smooths out fold-specific variance.
* **Logical Steps:**
    1. Instantiate ThreeTowerClassifier with all three modalities enabled.
    2. For each fold 1–5: load checkpoint, set eval mode, run get_embedding() on training and novel genes.
    3. Element-wise average the 5 embedding matrices per gene set.
* **I/O Variables:** Input: train_data, novel features, model checkpoints -> Output: train_emb_avg, novel_emb_avg (numpy arrays)

In [ ]:
import torch

# Add scripts directory to path so we can import model.py
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
sys.path.insert(0, os.path.abspath("."))
from model import ThreeTowerClassifier

# Convert feature arrays to tensors
train_npp_t = torch.tensor(train_data["npp"], dtype=torch.float32)
train_esm2_t = torch.tensor(train_data["esm2"], dtype=torch.float32)
train_ppi_t = torch.tensor(train_data["ppi"], dtype=torch.float32)
novel_npp_t = torch.tensor(novel_npp, dtype=torch.float32)
novel_esm2_t = torch.tensor(novel_esm2, dtype=torch.float32)
novel_ppi_t = torch.tensor(novel_ppi, dtype=torch.float32)

# Accumulate embeddings from each fold
n_folds = 5
train_emb_folds = []
novel_emb_folds = []

model = ThreeTowerClassifier(use_npp=True, use_esm2=True, use_ppi=True)

for fold in range(1, n_folds + 1):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"fusion_3tower_fold{fold}_best.pt")
    checkpoint = torch.load(ckpt_path, map_location="cpu", weights_only=True)
    model.load_state_dict(checkpoint["state_dict"])
    model.eval()
    logging.info(f"Fold {fold}: loaded checkpoint from {ckpt_path}")

    with torch.no_grad():
        # Embeddings for training genes
        train_emb = model.get_embedding(train_npp_t, train_esm2_t, train_ppi_t).numpy()
        train_emb_folds.append(train_emb)

        # Embeddings for novel candidates
        novel_emb = model.get_embedding(novel_npp_t, novel_esm2_t, novel_ppi_t).numpy()
        novel_emb_folds.append(novel_emb)

# Average embeddings across folds (element-wise mean)
train_emb_avg = np.mean(train_emb_folds, axis=0)
novel_emb_avg = np.mean(novel_emb_folds, axis=0)
logging.info(f"Averaged embeddings — training: {train_emb_avg.shape}, novel: {novel_emb_avg.shape}")

## UMAP Dimensionality Reduction
* **Objective (What):** Project the 64-dim averaged embeddings down to 2D with UMAP for visualization.
* **Rationale (Why):** UMAP preserves local neighbourhood structure better than t-SNE for this kind of multi-class clustering, and the 2D projection enables interpretable scatter plots.
* **Logical Steps:**
    1. Concatenate training and novel candidate embeddings into a single matrix.
    2. Fit UMAP on the combined matrix.
    3. Split the 2D coordinates back into training and novel sets.
* **I/O Variables:** Input: train_emb_avg, novel_emb_avg -> Output: umap_train, umap_novel (2D coordinates)

In [ ]:
import umap

# Concatenate training and novel embeddings
n_train = train_emb_avg.shape[0]
combined_emb = np.vstack([train_emb_avg, novel_emb_avg])
logging.info(f"Combined embedding matrix: {combined_emb.shape}")

# Fit UMAP on the combined matrix
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
umap_coords = reducer.fit_transform(combined_emb)
logging.info(f"UMAP projection complete: {umap_coords.shape}")

# Split back into training and novel coordinates
umap_train = umap_coords[:n_train]
umap_novel = umap_coords[n_train:]
logging.info(f"UMAP coordinates — training: {umap_train.shape}, novel: {umap_novel.shape}")

## UMAP Scatter Plot
* **Objective (What):** Create a publication-quality scatter plot of the UMAP projection.
* **Rationale (Why):** Visualizing the embedding space reveals module clustering quality and how novel candidates map to known modules.
* **Logical Steps:**
    1. Plot training genes colored by module label with module number at each cluster centroid.
    2. Overlay novel candidates as star markers with gene name labels.
    3. Add legend, title, and save at 300 dpi.
* **I/O Variables:** Input: umap_train, umap_novel, labels, novel_df -> Output: PNG figure

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from adjustText import adjust_text  # auto-resolves overlapping gene-name labels

# ── Style & global settings ────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')  # clean whitegrid background — swap for 'seaborn-v0_8-dark' or 'ggplot' if preferred

# ── Fixed 17-colour palette ────────────────────────────────────────────────
# Edit this list to change any module colour. Order maps to sorted unique module labels.
MODULE_COLORS = [
    '#1f77b4',  # module colour  0 — steel blue
    '#ff7f0e',  # module colour  1 — orange
    '#2ca02c',  # module colour  2 — green
    '#d62728',  # module colour  3 — brick red
    '#9467bd',  # module colour  4 — purple
    '#8c564b',  # module colour  5 — brown
    '#e377c2',  # module colour  6 — pink
    '#7f7f7f',  # module colour  7 — grey
    '#bcbd22',  # module colour  8 — olive
    '#17becf',  # module colour  9 — cyan
    '#aec7e8',  # module colour 10 — light blue
    '#ffbb78',  # module colour 11 — light orange
    '#98df8a',  # module colour 12 — light green
    '#ff9896',  # module colour 13 — salmon
    '#c5b0d5',  # module colour 14 — lavender
    '#c49c94',  # module colour 15 — tan
    '#f7b6d2',  # module colour 16 — light pink
]

# Prepare labels and colour mapping
train_labels = train_data['labels']  # integer module labels for 434 training genes
unique_modules = sorted(np.unique(train_labels))
n_modules = len(unique_modules)
module_color_map = {m: MODULE_COLORS[i % len(MODULE_COLORS)] for i, m in enumerate(unique_modules)}

# ── Create figure ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(
    figsize=(18, 13),  # figure width × height in inches — increase for more whitespace, decrease for compact
    facecolor='white',  # figure background colour
)

# ── Plot training genes ────────────────────────────────────────────────────
for m in unique_modules:
    mask = train_labels == m
    ax.scatter(
        umap_train[mask, 0],
        umap_train[mask, 1],
        s=60,              # training gene dot size — increase for larger dots, decrease if too crowded
        c=module_color_map[m],  # colour from the fixed palette — change MODULE_COLORS list to alter
        alpha=0.7,         # transparency of training dots — lower = more transparent, helps see overlap
        edgecolors='white',  # thin white outline separates overlapping dots
        linewidths=0.3,    # outline width — increase for thicker borders
        label=f'Module {m}',  # legend entry text — shown in the side legend
        zorder=2,          # draw order — training genes sit behind novel candidates
    )

    # Module centroid label
    centroid_x = umap_train[mask, 0].mean()  # x-centre of this module's cluster
    centroid_y = umap_train[mask, 1].mean()  # y-centre of this module's cluster
    ax.text(
        centroid_x,
        centroid_y,
        str(m),
        fontsize=11,       # centroid label font size — increase for larger numbers
        fontweight='bold', # bold makes centroid numbers stand out from scatter
        ha='center',       # horizontal alignment — keeps number centred on point
        va='center',       # vertical alignment — keeps number centred on point
        color='black',     # centroid label text colour
        bbox=dict(
            boxstyle='round,pad=0.3',  # rounded rectangle with 0.3 padding — increase pad for more whitespace
            facecolor='white',         # box fill colour
            edgecolor='gray',          # box border colour
            alpha=0.85,                # box opacity — lower = more see-through
            linewidth=0.8,             # box border thickness
        ),
        zorder=4,          # draw order — centroid labels sit above dots but below stars
    )

# ── Plot novel candidates (stars) ──────────────────────────────────────────
novel_modules = novel_df['predicted_module'].values
texts = []  # collect Text objects for adjustText to resolve overlaps

for i, (gene, mod) in enumerate(zip(novel_genes, novel_modules)):
    ax.scatter(
        umap_novel[i, 0],
        umap_novel[i, 1],
        s=250,             # novel candidate star size — make larger to stand out more
        c=module_color_map[mod],  # same colour as the predicted module
        marker='*',        # star marker distinguishes novel candidates from training dots
        edgecolors='black',  # black outline makes stars pop against coloured clusters
        linewidths=0.8,    # star outline width — increase for bolder outline
        zorder=5,          # draw order — stars render on top of everything except labels
        alpha=1.0,         # fully opaque so stars are always visible
    )

    # Gene name label (collected, not positioned yet — adjustText handles placement)
    t = ax.text(
        umap_novel[i, 0],
        umap_novel[i, 1],
        f' {gene}',        # leading space adds a tiny gap between star and label
        fontsize=8.5,      # gene label font size — increase for larger text
        fontweight='bold', # bold gene names for readability
        color='black',     # gene label text colour
        zorder=6,          # draw order — gene labels render above everything
    )
    texts.append(t)

# Resolve overlapping gene name labels automatically
# adjust_text(
#     texts,
#     arrowprops=dict(
#         arrowstyle='-',    # simple line connector — use '->' for an arrowhead
#         color='gray',      # connector line colour
#         lw=0.5,            # connector line width — increase for thicker arrows
#     ),

# Resolve overlapping gene name labels automatically and repel from points
adjust_text(
    texts,
    x=umap_novel[:, 0],       # explicitly pass x-coordinates of points to avoid
    y=umap_novel[:, 1],       # explicitly pass y-coordinates of points to avoid
    expand_points=(2.5, 2.5), # repel labels from data points more strongly
    expand_text=(2.5, 2.5),   # repel labels from each other more strongly
    force_text=(2, 2),    # apply additional force to separate text
    arrowprops=dict(
        arrowstyle='-',       # simple line connector
        color='gray',         # connector line colour
        lw=0.5,               # connector line width
    ),
)

# ── Legend ──────────────────────────────────────────────────────────────────
legend_patches = [
    mpatches.Patch(
        color=module_color_map[m],  # patch colour matches the module
        label=f'Module {m}',        # patch text
    )
    for m in unique_modules
]
ax.legend(
    handles=legend_patches,
    bbox_to_anchor=(1.01, 1.0),  # legend anchor — (1.01, 1) places it just outside upper-right
    loc='upper left',            # which corner of the legend box sits at the anchor point
    fontsize=9,                  # legend font size — decrease if legend is too large
    frameon=True,                # draw a box around the legend
    framealpha=0.9,              # legend box opacity
    edgecolor='lightgray',       # legend box border colour
    title='Modules',             # legend title text
    title_fontsize=10,           # legend title font size
)

# ── Title ──────────────────────────────────────────────────────────────────
fig.suptitle(
    'Fusion Embedding Space of Multi-Modal Protein Representations Recapitulates Inherited Retinal Disease Gene Module Structure',
    fontsize=18,        # main title font size
    fontweight='bold',  # bold main title
)
ax.set_title(
    'UMAP projection of 434 IRD training genes (circles, colored by phenotypic module) and top novel candidates (star)\nFeatures: evolutionary conservation (NPP), protein sequence context (ESM2), and protein–protein interaction network topology (PPI)',

    # insert your subtitle text

    fontsize=14,        # subtitle font size
    pad=18,             # padding between subtitle and axes
)

# ── Annotation in bottom-left corner ───────────────────────────────────────
fig.text(
    0.02,               # x position in figure coordinates (0 = left edge) — increase to shift right
    0.02,               # y position in figure coordinates (0 = bottom edge) — increase to shift up
    '\u2605 Novel candidate genes (top-5 per high-reliability module)',  # ★ unicode star
    fontsize=11,         # annotation font size
    color='gray',       # annotation text colour — use darker gray if hard to read
    style='italic',     # italic style distinguishes the caption from data labels
)

# ── Remove all axis spines and ticks ───────────────────────────────────────
for spine in ax.spines.values():
    spine.set_visible(True)   # show all four spines to create a frame
    spine.set_color('lightgray')  # set frame color
    spine.set_linewidth(1.0)  # set frame line thickness

ax.set_xticks([])  # remove x-axis tick marks
ax.set_yticks([])  # remove y-axis tick marks
ax.grid(False)     # disable the whitegrid gridlines inside the plot area

plt.tight_layout()  # auto-adjust subplot spacing so legend and labels are not clipped

# ── Save figure ────────────────────────────────────────────────────────────
output_path = os.path.join(OUTPUT_DIR, 'umap_fusion_3tower.png')
fig.savefig(
    output_path,
    dpi=300,              # output resolution — 300 is publication quality; use 150 for quick previews
    bbox_inches='tight',  # crop whitespace around the figure
    facecolor='white',    # saved image background colour
)
logging.info(f'Figure saved to {output_path}')
plt.show()
logging.info('UMAP visualization complete.')
